## START

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
# -- takes a few mins to run first time - sentence trainsformers is about 5GB in size...
# also hugging face complains - I need HF_TOKEN to enable higher rate limits and faster downloads.

# The first time I run this, it downloads the model (~80 MB) and the tokenizer from HuggingFace. 
# The tokenizer turns text into something the model can read. After that, both load from a local cache.

# Trying it with simple examples
# Let's see how embeddings work on a few examples.

# We'll start with a query:

q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
# v1 # prints embedding vector like array([ 2.13903598e-02, -7.39799812e-02,  1.42071780e-03,  2.13816296e-02, ...

# v1 is a vector, an array of 384 numbers. Each number stands for some concept the model learned. 
# We can't read off what any one of them means. But two vectors with similar values point to texts about similar things.


In [6]:
# Encode our simplest document:

d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [8]:
# Next, we compare the query against the document using dot product:
# this is cosine similarity metrics - dist between embedding vectors - but we use matrix multiplication for simplicity here :

v1.dot(dv)

# We get 0.32. -- np.float32(0.32332408)

np.float32(0.32332408)

In [10]:
# Now we try an unrelated query:

q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

# This time the similarity with the document should be much smaller:

v2.dot(dv)

# And we get 0.01. -- np.float32(0.019730505) - very small because Q and A are symantically dissimilar

np.float32(0.019730505)

In [13]:
q2 = "How much is the fish?"
v2 = model.encode(q2)

# totally unrelated Q - the similarity with the document should be smaller:

v2.dot(dv)

np.float32(-0.073930666)

In [14]:
# EXPLANATION:

# The first score for q1 vs d (0.32) is higher, so that query is more similar to the document about registration. 
# The second score for q2 vs d sits near 0, because installing Docker has nothing to do with registration. 
# A score near 0 means the two vectors are about as different as they can be.

# That's the whole idea behind vector search: similar texts get similar vectors, and a dot product tells us how similar.

# Cosine similarity
# The all-MiniLM-L6-v2 model outputs normalized vectors - vectors with unit length. When both vectors are normalized, 
# the dot product equals cosine similarity. That's why the model documentation says it "uses cosine similarity."

# Cosine similarity measures the angle between two vectors, ignoring their length:

# 1.0 = same direction (similar)
# 0.0 = perpendicular (unrelated)
# -1.0 = opposite direction (opposite meaning)
# Formally, if theta is the angle between two vectors, cosine similarity is cos(theta):

# cos(0) = 1 - vectors point in the same direction
# cos(90) = 0 - vectors are perpendicular
# cos(180) = -1 - vectors point in opposite directions
# Because our vectors are normalized, the dot product gives us cosine similarity directly. This is why we can use v1.dot(dv) to compare texts.

# In practice, we rarely get cosine similarity below 0. The embedding model maps text to a region of the vector space 
# where most vectors have positive components. There's no concept of "opposite meaning" that maps to a vector pointing the other way.

## Embedding Our Dataset

In [18]:
# Loading the data
# In module 1 we created ingest.py for loading the FAQ data.

# Download it into your project:
# choco install wget
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
# We use it here:

from ingest import load_faq_data

documents = load_faq_data()

In [19]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [20]:
len(documents)

1350

In [22]:
# Generating embeddings
# Each document is a Python dictionary with a question and an answer. We embed both together. That way a query can match against the question text and the answer text in our index.

# Build one text per document:

texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

texts[10] # we combined Q and A - see documents[10] from above

'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [23]:
# Now we generate the embeddings. We have about 1200 texts here. We won't hand the model all of them at once. 
# That takes a long time, and we can't see what's happening inside. Instead we split them into batches.

# First we import tqdm so we can watch the progress:
# uv add tqdm

from tqdm.auto import tqdm
# Next we chunk the dataset into batches of 50 and encode each batch:

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)
# We end up with 1350 vectors. On a GPU this is fast. Most of us run on Codespaces without a GPU, so it takes a bit, but it's a one-off.


  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [24]:
# We turn them into a 2-dimensional array (matrix) where
# rows are documents (vectors)
# columns are dimensions of the vectors

import numpy as np
X = np.array(vectors)
X.shape

# Calling X.shape returns (1350, 384) - number of documents vs number of dimensions.

(1350, 384)